# IDK time benchmarking

Run each combination of the `_GRID` parameters and time **fit**, **embed**, and **similarity**. Data generation and model construction are outside the timers. Results stay in memory and appear in the table below.

Use the project's `nb` environment. Large grid settings can take a long time.


In [1]:
from itertools import product
from time import perf_counter

import numpy as np
from IPython.display import Markdown, display

from pyidk import IsolationDistributionalKernel, IsolationKernel, SequenceBatch

SEED = 42
N_PARTITIONS_GRID = [25, 100, 250]
PSI_FRACTIONS_GRID = [0.0005, 0.005]
SAMPLE_COUNTS_GRID = [2**10, 2**16, 2**18, 2**20]  # 1K, 65K, 262K, 1M
FEATURE_COUNTS_GRID = [2, 4, 16]
N_DISTRIBUTIONS = 16  # Independent groups/episodes
REPEATS = 1
CHUNK_SIZE = 4096


def psi_for(n_samples: int, psi_fraction: float) -> int:
    return min(n_samples, max(2, int(np.ceil(psi_fraction * n_samples))))

## Terminology

- **N** is the total number of observations
- **d** is input feature dimension
- **t** is the number of partitions
- **psi** is the number of centers sampled per partition

In [2]:
configurations = list(
    product(
        SAMPLE_COUNTS_GRID,
        FEATURE_COUNTS_GRID,
        N_PARTITIONS_GRID,
        PSI_FRACTIONS_GRID,
    )
)
print(f"{len(configurations)} configurations, {REPEATS} repetition(s) each.")

72 configurations, 1 repetition(s) each.


## Execution Time Contributors

Here are the steps that contribute to time complexity:

- **Fit** 
- **Embed** 
- **Similarity** 

Some research needs to be conducted to determine precise complexity.

## Benchmarking Functions


In [3]:
def make_dataset(n_samples: int, n_features: int) -> SequenceBatch:
    if n_samples < N_DISTRIBUTIONS:
        raise ValueError("Every distribution must contain at least one observation.")
    rng = np.random.default_rng(np.random.SeedSequence([SEED, n_samples, n_features]))
    values = rng.standard_normal((n_samples, n_features))
    # Balanced groups cover all rows even when division has a remainder.
    offsets = np.arange(N_DISTRIBUTIONS + 1, dtype=np.int64) * n_samples // N_DISTRIBUTIONS
    return SequenceBatch(values, offsets)


def benchmark_once(data: SequenceBatch, psi: int, n_partitions: int) -> dict[str, float]:
    model = IsolationDistributionalKernel(
        IsolationKernel(
            n_partitions=n_partitions,
            samples_per_partition=psi,
            random_state=SEED,
            chunk_size=CHUNK_SIZE,
        )
    )
    start = perf_counter()
    model.fit(data)
    fitted = perf_counter()
    embeddings = model.transform(data)
    embedded = perf_counter()
    model.similarity(embeddings)
    finished = perf_counter()

    return {
        "fit_s": fitted - start,
        "embed_s": embedded - fitted,
        "similarity_s": finished - embedded,
        "total_s": finished - start,
    }

## Run the grid

Each repetition uses a fresh model. The loop prints elapsed time after each configuration.


In [4]:
records = []
for _ in range(REPEATS):
    for n, d, t, fraction in configurations:
        data = make_dataset(n, d)
        psi = psi_for(n, fraction)
        result = benchmark_once(data, psi, t)
        records.append({"n": n, "d": d, "t": t, "fraction": fraction, "psi": psi, **result})
        print(f"N={n:,}, d={d}, t={t}, psi={psi:,}: {result['total_s']:.3f} s", flush=True)

N=1,024, d=2, t=25, psi=2: 0.002 s
N=1,024, d=2, t=25, psi=6: 0.002 s
N=1,024, d=2, t=100, psi=2: 0.005 s
N=1,024, d=2, t=100, psi=6: 0.006 s
N=1,024, d=2, t=250, psi=2: 0.011 s
N=1,024, d=2, t=250, psi=6: 0.015 s
N=1,024, d=4, t=25, psi=2: 0.002 s
N=1,024, d=4, t=25, psi=6: 0.002 s
N=1,024, d=4, t=100, psi=2: 0.005 s
N=1,024, d=4, t=100, psi=6: 0.006 s
N=1,024, d=4, t=250, psi=2: 0.012 s
N=1,024, d=4, t=250, psi=6: 0.014 s
N=1,024, d=16, t=25, psi=2: 0.002 s
N=1,024, d=16, t=25, psi=6: 0.002 s
N=1,024, d=16, t=100, psi=2: 0.006 s
N=1,024, d=16, t=100, psi=6: 0.007 s
N=1,024, d=16, t=250, psi=2: 0.012 s
N=1,024, d=16, t=250, psi=6: 0.016 s
N=65,536, d=2, t=25, psi=33: 0.099 s
N=65,536, d=2, t=25, psi=328: 0.646 s
N=65,536, d=2, t=100, psi=33: 0.380 s
N=65,536, d=2, t=100, psi=328: 2.490 s
N=65,536, d=2, t=250, psi=33: 0.914 s
N=65,536, d=2, t=250, psi=328: 6.197 s
N=65,536, d=4, t=25, psi=33: 0.106 s
N=65,536, d=4, t=25, psi=328: 0.775 s
N=65,536, d=4, t=100, psi=33: 0.415 s
N=65,536, 

## Results table

Times are medians across repetitions, in seconds. With one repetition, each entry is the measured time.


In [5]:
rows = []
for n, d, t, fraction in configurations:
    runs = [r for r in records if (r["n"], r["d"], r["t"], r["fraction"]) == (n, d, t, fraction)]
    if not runs:
        continue
    times = [
        np.median([r[key] for r in runs]) for key in ["fit_s", "embed_s", "similarity_s", "total_s"]
    ]
    rows.append(
        f"| {n:,} | {d} | {t} | {fraction:g} | {psi_for(n, fraction):,} | "
        + " | ".join(f"{value:.4f}" for value in times)
        + " |"
    )
display(
    Markdown(
        "| N | d | t | Fraction | psi | Fit s | Embed s | Similarity s | Total s |\n"
        "|--:|--:|--:|--:|--:|--:|--:|--:|--:|\n" + "\n".join(rows)
    )
)

| N | d | t | Fraction | psi | Fit s | Embed s | Similarity s | Total s |
|--:|--:|--:|--:|--:|--:|--:|--:|--:|
| 1,024 | 2 | 25 | 0.0005 | 2 | 0.0007 | 0.0015 | 0.0001 | 0.0024 |
| 1,024 | 2 | 25 | 0.005 | 6 | 0.0007 | 0.0014 | 0.0001 | 0.0022 |
| 1,024 | 2 | 100 | 0.0005 | 2 | 0.0011 | 0.0037 | 0.0001 | 0.0049 |
| 1,024 | 2 | 100 | 0.005 | 6 | 0.0012 | 0.0043 | 0.0002 | 0.0057 |
| 1,024 | 2 | 250 | 0.0005 | 2 | 0.0023 | 0.0090 | 0.0002 | 0.0114 |
| 1,024 | 2 | 250 | 0.005 | 6 | 0.0021 | 0.0121 | 0.0005 | 0.0148 |
| 1,024 | 4 | 25 | 0.0005 | 2 | 0.0003 | 0.0014 | 0.0001 | 0.0017 |
| 1,024 | 4 | 25 | 0.005 | 6 | 0.0005 | 0.0016 | 0.0001 | 0.0021 |
| 1,024 | 4 | 100 | 0.0005 | 2 | 0.0011 | 0.0041 | 0.0001 | 0.0053 |
| 1,024 | 4 | 100 | 0.005 | 6 | 0.0012 | 0.0049 | 0.0002 | 0.0064 |
| 1,024 | 4 | 250 | 0.0005 | 2 | 0.0023 | 0.0092 | 0.0002 | 0.0117 |
| 1,024 | 4 | 250 | 0.005 | 6 | 0.0017 | 0.0118 | 0.0005 | 0.0140 |
| 1,024 | 16 | 25 | 0.0005 | 2 | 0.0003 | 0.0014 | 0.0001 | 0.0018 |
| 1,024 | 16 | 25 | 0.005 | 6 | 0.0003 | 0.0017 | 0.0001 | 0.0021 |
| 1,024 | 16 | 100 | 0.0005 | 2 | 0.0009 | 0.0046 | 0.0001 | 0.0056 |
| 1,024 | 16 | 100 | 0.005 | 6 | 0.0012 | 0.0057 | 0.0002 | 0.0071 |
| 1,024 | 16 | 250 | 0.0005 | 2 | 0.0021 | 0.0100 | 0.0002 | 0.0123 |
| 1,024 | 16 | 250 | 0.005 | 6 | 0.0022 | 0.0137 | 0.0005 | 0.0164 |
| 65,536 | 2 | 25 | 0.0005 | 33 | 0.0003 | 0.0980 | 0.0003 | 0.0986 |
| 65,536 | 2 | 25 | 0.005 | 328 | 0.0025 | 0.6405 | 0.0026 | 0.6456 |
| 65,536 | 2 | 100 | 0.0005 | 33 | 0.0014 | 0.3777 | 0.0011 | 0.3802 |
| 65,536 | 2 | 100 | 0.005 | 328 | 0.0088 | 2.4691 | 0.0124 | 2.4903 |
| 65,536 | 2 | 250 | 0.0005 | 33 | 0.0017 | 0.9101 | 0.0027 | 0.9144 |
| 65,536 | 2 | 250 | 0.005 | 328 | 0.0220 | 6.1379 | 0.0368 | 6.1966 |
| 65,536 | 4 | 25 | 0.0005 | 33 | 0.0003 | 0.1059 | 0.0003 | 0.1064 |
| 65,536 | 4 | 25 | 0.005 | 328 | 0.0031 | 0.7690 | 0.0025 | 0.7746 |
| 65,536 | 4 | 100 | 0.0005 | 33 | 0.0008 | 0.4136 | 0.0011 | 0.4155 |
| 65,536 | 4 | 100 | 0.005 | 328 | 0.0123 | 3.0866 | 0.0127 | 3.1116 |
| 65,536 | 4 | 250 | 0.0005 | 33 | 0.0018 | 1.0589 | 0.0027 | 1.0635 |
| 65,536 | 4 | 250 | 0.005 | 328 | 0.0311 | 7.7257 | 0.0383 | 7.7951 |
| 65,536 | 16 | 25 | 0.0005 | 33 | 0.0003 | 0.1885 | 0.0003 | 0.1891 |
| 65,536 | 16 | 25 | 0.005 | 328 | 0.0077 | 1.6845 | 0.0025 | 1.6947 |
| 65,536 | 16 | 100 | 0.0005 | 33 | 0.0010 | 0.7500 | 0.0011 | 0.7520 |
| 65,536 | 16 | 100 | 0.005 | 328 | 0.0305 | 6.7269 | 0.0120 | 6.7693 |
| 65,536 | 16 | 250 | 0.0005 | 33 | 0.0024 | 1.8831 | 0.0027 | 1.8882 |
| 65,536 | 16 | 250 | 0.005 | 328 | 0.0776 | 16.9943 | 0.0363 | 17.1083 |
| 262,144 | 2 | 25 | 0.0005 | 132 | 0.0006 | 1.0535 | 0.0011 | 1.0553 |
| 262,144 | 2 | 25 | 0.005 | 1,311 | 0.0310 | 10.6870 | 0.0144 | 10.7324 |
| 262,144 | 2 | 100 | 0.0005 | 132 | 0.0024 | 4.3572 | 0.0046 | 4.3641 |
| 262,144 | 2 | 100 | 0.005 | 1,311 | 0.1266 | 42.1402 | 0.0749 | 42.3417 |
| 262,144 | 2 | 250 | 0.0005 | 132 | 0.0054 | 10.6313 | 0.0138 | 10.6505 |
| 262,144 | 2 | 250 | 0.005 | 1,311 | 0.3072 | 105.5432 | 0.3012 | 106.1517 |
| 262,144 | 4 | 25 | 0.0005 | 132 | 0.0008 | 1.3279 | 0.0011 | 1.3298 |
| 262,144 | 4 | 25 | 0.005 | 1,311 | 0.0435 | 12.8411 | 0.0129 | 12.8976 |
| 262,144 | 4 | 100 | 0.0005 | 132 | 0.0030 | 5.3249 | 0.0045 | 5.3324 |
| 262,144 | 4 | 100 | 0.005 | 1,311 | 0.1724 | 51.4721 | 0.0790 | 51.7235 |
| 262,144 | 4 | 250 | 0.0005 | 132 | 0.0071 | 13.3430 | 0.0138 | 13.3639 |
| 262,144 | 4 | 250 | 0.005 | 1,311 | 0.4306 | 128.7733 | 0.3321 | 129.5359 |
| 262,144 | 16 | 25 | 0.0005 | 132 | 0.0018 | 2.7713 | 0.0011 | 2.7742 |
| 262,144 | 16 | 25 | 0.005 | 1,311 | 0.1212 | 28.7131 | 0.0126 | 28.8469 |
| 262,144 | 16 | 100 | 0.0005 | 132 | 0.0064 | 11.1809 | 0.0044 | 11.1917 |
| 262,144 | 16 | 100 | 0.005 | 1,311 | 0.4782 | 111.4211 | 0.0689 | 111.9682 |
| 262,144 | 16 | 250 | 0.0005 | 132 | 0.0152 | 27.7468 | 0.0136 | 27.7756 |
| 262,144 | 16 | 250 | 0.005 | 1,311 | 1.2025 | 278.7606 | 0.2908 | 280.2539 |
| 1,048,576 | 2 | 25 | 0.0005 | 525 | 0.0056 | 16.4773 | 0.0045 | 16.4874 |
| 1,048,576 | 2 | 25 | 0.005 | 5,243 | 0.4999 | 162.2861 | 0.0788 | 162.8648 |
| 1,048,576 | 2 | 100 | 0.0005 | 525 | 0.0232 | 66.0264 | 0.0245 | 66.0740 |
| 1,048,576 | 2 | 100 | 0.005 | 5,243 | 1.9822 | 649.5804 | 0.7337 | 652.2963 |
| 1,048,576 | 2 | 250 | 0.0005 | 525 | 0.0549 | 165.4035 | 0.0822 | 165.5407 |
| 1,048,576 | 2 | 250 | 0.005 | 5,243 | 4.9737 | 1623.7069 | 2.1299 | 1630.8105 |
| 1,048,576 | 4 | 25 | 0.0005 | 525 | 0.0078 | 20.4896 | 0.0045 | 20.5020 |
| 1,048,576 | 4 | 25 | 0.005 | 5,243 | 0.6961 | 202.7124 | 0.0729 | 203.4814 |
| 1,048,576 | 4 | 100 | 0.0005 | 525 | 0.0302 | 80.7624 | 0.0236 | 80.8162 |
| 1,048,576 | 4 | 100 | 0.005 | 5,243 | 2.7860 | 809.9928 | 0.6519 | 813.4307 |
| 1,048,576 | 4 | 250 | 0.0005 | 525 | 0.0754 | 202.0946 | 0.0814 | 202.2514 |
| 1,048,576 | 4 | 250 | 0.005 | 5,243 | 6.9624 | 2025.2439 | 2.1370 | 2034.3432 |
| 1,048,576 | 16 | 25 | 0.0005 | 525 | 0.0197 | 44.1274 | 0.0046 | 44.1516 |
| 1,048,576 | 16 | 25 | 0.005 | 5,243 | 1.8823 | 439.7965 | 0.0746 | 441.7535 |
| 1,048,576 | 16 | 100 | 0.0005 | 525 | 0.0792 | 175.9780 | 0.0237 | 176.0809 |
| 1,048,576 | 16 | 100 | 0.005 | 5,243 | 7.6507 | 1759.3956 | 0.5553 | 1767.6016 |
| 1,048,576 | 16 | 250 | 0.0005 | 525 | 0.1966 | 439.9475 | 0.0880 | 440.2321 |
| 1,048,576 | 16 | 250 | 0.005 | 5,243 | 18.8318 | 6170.2633 | 2.0342 | 6191.1293 |